# 05 — Results Visualisation: PGD · IBP · MILP Comparison

This notebook:
1. **Verifies correctness** of each method (PGD, IBP, MILP) against their theoretical guarantees.
2. **Runs all three methods** on the fixed 100-sample MNIST evaluation split.
3. **Produces comparative visualisations** — bar charts, heatmaps, margin plots, epsilon-sensitivity curves.
4. **Summarises insights** relevant to the thesis.

---
### Theoretical relationships (used as correctness checks)

| Claim | Reason |
|---|---|
| `PGD success` ⟹ `MILP FALSIFIED` | PGD finds a real adversarial example |
| `IBP VERIFIED` ⟹ `MILP VERIFIED` | IBP is sound (over-approximates reachable set) |
| `IBP VERIFIED` ⟹ `PGD fails` | If no adversarial example exists, no attack can succeed |
| `MILP VERIFIED` ⟹ `PGD fails` | MILP is complete (exact), so truly safe |
| IBP non-verified rate ≥ MILP non-verified rate | IBP is conservative, MILP is exact |

**Prerequisites:** Run `01_train_mnist.ipynb` first, then optionally run `02` and `03`.
This notebook will re-run PGD, IBP, and MILP if CSV results are not found.

In [ ]:
!pip install -q torch torchvision numpy pandas ortools matplotlib seaborn

## 1 — Full library code (models, data, PGD, IBP, MILP)

In [ ]:
from __future__ import annotations
import json, os, random, time
from dataclasses import asdict, dataclass
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim, Tensor
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ── Seed ──────────────────────────────────────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

# ── Data ──────────────────────────────────────────────────────────────────────
def get_mnist_datasets(data_dir="data"):
    tfm = transforms.ToTensor()
    return (datasets.MNIST(str(data_dir), train=True,  download=True, transform=tfm),
            datasets.MNIST(str(data_dir), train=False, download=True, transform=tfm))

def make_loader(ds, batch_size, shuffle, seed=1234):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, generator=g, drop_last=False)

# ── Evaluation split ──────────────────────────────────────────────────────────
@dataclass(frozen=True)
class Split:
    seed: int
    indices: list[int]

def load_split(path) -> Split:
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return Split(seed=int(obj["seed"]), indices=[int(i) for i in obj["indices"]])

def ensure_split(path, seed=1234, n=100) -> Split:
    p = Path(path)
    if p.exists(): return load_split(p)
    idxs = random.Random(seed).sample(range(10_000), n)
    split = Split(seed=seed, indices=idxs)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps({"seed": split.seed, "indices": split.indices}, indent=2)+"\n", encoding="utf-8")
    return split

# ── Models ────────────────────────────────────────────────────────────────────
class MnistMlp(nn.Module):
    def __init__(self, in_dim=784, h1=128, h2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(int(in_dim), int(h1))
        self.fc2 = nn.Linear(int(h1), int(h2))
        self.fc3 = nn.Linear(int(h2), int(num_classes))
        self.relu = nn.ReLU()
    def forward(self, x):
        if x.ndim == 4: x = x.view(x.shape[0], -1)
        return self.fc3(self.relu(self.fc2(self.relu(self.fc1(x)))))
    def linear_layers(self): return [self.fc1, self.fc2, self.fc3]

# ── Checkpoint ────────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str; model_kwargs: dict; run_name: str

def save_checkpoint(path, model, meta, metrics=None):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    payload = {"meta": asdict(meta), "model_state_dict": model.state_dict(), "metrics": metrics or {}}
    torch.save(payload, str(p))

def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(model_type=str(raw["model_type"]),
                          model_kwargs=dict(raw.get("model_kwargs", {})),
                          run_name=str(raw.get("run_name", "run")))
    model = MnistMlp(**meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})

print("Core library loaded")

In [ ]:
# ── PGD attack ────────────────────────────────────────────────────────────────
def pgd_linf(model, x0, y, eps, steps, step_size, random_start=True):
    """Projected Gradient Descent L-inf attack."""
    model.eval()
    x0 = x0.detach()
    x  = x0.clone()
    if random_start:
        lo = (x0 - eps).clamp(0., 1.); hi = (x0 + eps).clamp(0., 1.)
        x  = (x + (2*torch.rand_like(x)-1)*eps).clamp(lo, hi)
    for _ in range(int(steps)):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), y, reduction="sum")
        grad = torch.autograd.grad(loss, x)[0]
        with torch.no_grad():
            lo = (x0 - eps).clamp(0., 1.); hi = (x0 + eps).clamp(0., 1.)
            x  = (x + step_size * grad.sign()).clamp(lo, hi)
        x = x.detach()
    return x

# ── IBP standalone verifier ───────────────────────────────────────────────────
def compute_ibp_bounds(model, x_center: Tensor, eps: float) -> dict:
    """
    Interval Bound Propagation through the MLP.
    Returns per-layer pre-activation bounds [a_l_list, a_u_list].
    This is the SAME computation used internally by the MILP encoder
    (see bounds_ibp.py), exposed here as a standalone verifier.
    """
    x0  = x_center.detach().view(-1)
    x_l = torch.clamp(x0 - eps, 0., 1.)
    x_u = torch.clamp(x0 + eps, 0., 1.)
    h_l, h_u = [x_l], [x_u]
    a_l_list, a_u_list = [], []
    for li, layer in enumerate(model.linear_layers()):
        W = layer.weight.detach()
        b = layer.bias.detach()
        Wp = W.clamp(min=0.); Wn = W.clamp(max=0.)
        al = Wp @ h_l[-1] + Wn @ h_u[-1] + b
        au = Wp @ h_u[-1] + Wn @ h_l[-1] + b
        a_l_list.append(al); a_u_list.append(au)
        if li < len(model.linear_layers()) - 1:
            h_l.append(al.clamp(min=0.)); h_u.append(au.clamp(min=0.))
    return {"x_l": x_l, "x_u": x_u, "a_l_list": a_l_list, "a_u_list": a_u_list}


def ibp_verify_point(model, x: Tensor, y: int, eps: float):
    """
    IBP standalone verification.
    Decision rule: for every adversarial class c != y, the worst-case margin
      logit_c - logit_y  is bounded above by  a_u[-1][c] - a_l[-1][y].
    If this upper bound is <= 0 for ALL c, the sample is provably VERIFIED.
    Otherwise it is INCONCLUSIVE (IBP is sound but incomplete).

    Returns (status: str, worst_ibp_margin: float)
    """
    bounds   = compute_ibp_bounds(model, x, eps)
    logit_l  = bounds["a_l_list"][-1]   # lower bounds on logits
    logit_u  = bounds["a_u_list"][-1]   # upper bounds on logits
    y_int    = int(y)
    n_cls    = logit_l.shape[0]
    # Worst-case adversarial margin = max_{c!=y} (ub_logit_c - lb_logit_y)
    worst_margin = max(
        float(logit_u[c].item()) - float(logit_l[y_int].item())
        for c in range(n_cls) if c != y_int
    )
    status = "VERIFIED" if worst_margin <= 0.0 else "INCONCLUSIVE"
    return status, worst_margin

print("PGD and IBP verifier defined")

In [ ]:
# ── MILP types & LinExpr algebra ──────────────────────────────────────────────
class VerifStatus(str, Enum):
    VERIFIED  = "VERIFIED"
    FALSIFIED = "FALSIFIED"
    TIMEOUT   = "TIMEOUT"
    ERROR     = "ERROR"

@dataclass
class VerifResult:
    status: VerifStatus; worst_margin: float|None
    solve_time: float|None; solver_name: str; adv_example: list[float]|None

@dataclass
class LinExpr:
    terms: Dict[Any, float]; const: float = 0.0

def e_c(c):              return LinExpr(terms={}, const=float(c))
def e_v(v, k=1.):        return LinExpr(terms={v: float(k)}, const=0.)
def e_mul(a, s):         return LinExpr({v: c*float(s) for v,c in a.terms.items()}, a.const*float(s))
def e_add(a, b):
    t = dict(a.terms)
    for v, c in b.terms.items(): t[v] = t.get(v, 0.) + c
    return LinExpr(t, a.const + b.const)
def e_sub(a, b): return e_add(a, e_mul(b, -1.))

# ── OR-Tools CBC backend ──────────────────────────────────────────────────────
class CbcBackend:
    solver_name = "CBC"
    def __init__(self):
        from ortools.linear_solver import pywraplp
        self._pw = pywraplp
        s = pywraplp.Solver.CreateSolver("CBC") or pywraplp.Solver.CreateSolver("CBC_MIXED_INTEGER_PROGRAMMING")
        if s is None: raise RuntimeError("Cannot create CBC solver")
        self.s = s; self._obj = None
    def add_var(self, lb, ub, binary=False):
        return self.s.IntVar(0.,1.,"") if binary else self.s.NumVar(float(lb),float(ub),"")
    def add_binary(self): return self.add_var(0,1,True)
    def _e(self, e):
        lin = self.s.Sum([c*v for v,c in e.terms.items()])
        return lin + float(e.const) if e.const else lin
    def add_le(self,l,r): self.s.Add(self._e(e_sub(l,r))<=0.)
    def add_ge(self,l,r): self.s.Add(self._e(e_sub(l,r))>=0.)
    def add_eq(self,l,r): self.s.Add(self._e(e_sub(l,r))==0.)
    def set_objective_max(self, e): self._obj=e; self.s.Maximize(self._e(e))
    def solve(self, time_limit_s=None):
        if time_limit_s: self.s.set_time_limit(int(max(time_limit_s,0.)*1000))
        t0=time.time(); st=self.s.Solve(); el=time.time()-t0
        pw=self._pw
        sm={pw.Solver.OPTIMAL:"OPTIMAL",pw.Solver.FEASIBLE:"FEASIBLE",
            pw.Solver.INFEASIBLE:"INFEASIBLE",pw.Solver.UNBOUNDED:"UNBOUNDED"}
        obj=float(self.s.Objective().Value()) if self._obj and st in(pw.Solver.OPTIMAL,pw.Solver.FEASIBLE) else None
        return sm.get(st,"UNKNOWN"), obj, el
    def get_value(self, v): return float(v.solution_value())

# ── MILP encoder ──────────────────────────────────────────────────────────────
def encode_milp(backend, model, x_center, eps):
    """Encode MLP as MILP with big-M ReLU constraints (IBP-tight bounds)."""
    bounds = compute_ibp_bounds(model, x_center, eps)
    x_l, x_u = bounds["x_l"], bounds["x_u"]
    x0 = x_center.detach().view(-1)
    in_vars = [backend.add_var(float(x_l[i]),float(x_u[i])) for i in range(x0.numel())]
    linears = model.linear_layers()
    prev = in_vars; logit_vars = []
    for li, layer in enumerate(linears):
        W,b = layer.weight.detach(), layer.bias.detach()
        a_vars = [backend.add_var(-1e9,1e9) for _ in range(W.shape[0])]
        for i in range(W.shape[0]):
            rhs = e_c(float(b[i]))
            for j in range(W.shape[1]):
                c = float(W[i,j])
                if c != 0.: rhs = e_add(rhs, e_mul(e_v(prev[j]),c))
            backend.add_eq(e_v(a_vars[i]), rhs)
        if li == len(linears)-1:
            logit_vars = a_vars
        else:
            al,au = bounds["a_l_list"][li], bounds["a_u_list"][li]
            h_vars = []
            for i,av in enumerate(a_vars):
                l,u = float(al[i]),float(au[i])
                if u <= 0.:
                    h = backend.add_var(0.,0.)
                elif l >= 0.:
                    h = backend.add_var(l,u); backend.add_eq(e_v(h),e_v(av))
                else:
                    h=backend.add_var(0.,u); bb=backend.add_binary()
                    backend.add_ge(e_v(h),e_c(0.)); backend.add_ge(e_v(h),e_v(av))
                    omb=e_add(e_c(1.),e_mul(e_v(bb),-1.))
                    backend.add_le(e_v(h),e_add(e_v(av),e_mul(omb,-l)))
                    backend.add_le(e_v(h),e_mul(e_v(bb),u))
                    backend.add_ge(e_v(av),e_c(l)); backend.add_le(e_v(av),e_c(u))
                h_vars.append(h)
            prev = h_vars
    return {"input_vars": in_vars, "logit_vars": logit_vars}

def milp_verify_point(model, x: Tensor, y: int, eps: float, time_limit_s=30.) -> VerifResult:
    """Exact MILP robustness verification."""
    x = x.detach()
    worst = float("-inf"); best_adv = None; t_acc = 0.; last_solver = ""
    for c in [ci for ci in range(10) if ci != int(y)]:
        bk = CbcBackend()
        enc = encode_milp(bk, model, x, eps)
        bk.set_objective_max(e_sub(e_v(enc["logit_vars"][c]), e_v(enc["logit_vars"][int(y)])))
        ss, obj, t = bk.solve(time_limit_s=time_limit_s)
        last_solver = bk.solver_name; t_acc += float(t)
        if ss in {"OPTIMAL","FEASIBLE"}: s = VerifStatus.FALSIFIED
        elif ss in {"INFEASIBLE","UNBOUNDED"}: s = VerifStatus.VERIFIED
        elif ss == "TIME_LIMIT": s = VerifStatus.TIMEOUT
        else: s = VerifStatus.ERROR
        if s in {VerifStatus.ERROR, VerifStatus.TIMEOUT}:
            return VerifResult(status=s, worst_margin=None, solve_time=t_acc,
                               solver_name=last_solver, adv_example=None)
        if obj is not None and obj > worst:
            worst = float(obj)
            if s == VerifStatus.FALSIFIED:
                best_adv = [bk.get_value(v) for v in enc["input_vars"]]
    if worst <= 0.:
        return VerifResult(VerifStatus.VERIFIED, worst, t_acc, last_solver, None)
    return VerifResult(VerifStatus.FALSIFIED, worst, t_acc, last_solver, best_adv)

print("MILP verifier defined")

## 2 — Configuration

In [ ]:
# ── Edit these as needed ───────────────────────────────────────────────────────
CKPT_PATH    = "runs/mlp_mnist/model.pt"
SUBSET_PATH  = "assets/splits/mnist_eval_100.json"
DATA_DIR     = "data"
SEED         = 1234

# PGD settings
EPS          = 0.03
PGD_STEPS    = 40
STEP_SIZE    = 0.01
BATCH_SIZE   = 64

# MILP settings (keep MAX_MILP_SAMPLES small — it's slow)
MAX_MILP_SAMPLES = 10
TIME_LIMIT       = 30.0

# Epsilon values for sensitivity analysis
EPS_SWEEP = [0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)
print(f"Device: {DEVICE} | eps={EPS} | MILP samples={MAX_MILP_SAMPLES}")

## 3 — Load or train model

In [ ]:
from tqdm import tqdm

def train_quick(device, data_dir, seed, epochs=3):
    """Quick MLP training — only called if no checkpoint exists."""
    print("No checkpoint found. Training MLP (3 epochs)...")
    train_ds, test_ds = get_mnist_datasets(data_dir)
    train_loader = make_loader(train_ds, 128, True,  seed)
    test_loader  = make_loader(test_ds,  128, False, seed)
    model = MnistMlp().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs+1):
        model.train(); corr = tot = 0
        for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x); loss = F.cross_entropy(logits, y)
            loss.backward(); opt.step()
            corr += int((logits.argmax(1)==y).sum()); tot += int(y.numel())
        model.eval(); c2 = t2 = 0
        with torch.no_grad():
            for x,y in test_loader:
                x,y=x.to(device),y.to(device)
                c2+=int((model(x).argmax(1)==y).sum()); t2+=int(y.numel())
        print(f"  epoch={epoch}  train_acc={corr/tot:.4f}  test_acc={c2/t2:.4f}")
    meta = CheckpointMeta("mlp", {"in_dim":784,"h1":128,"h2":64,"num_classes":10}, "mlp_mnist")
    save_checkpoint(CKPT_PATH, model, meta)
    return model, meta

if Path(CKPT_PATH).exists():
    model, meta, ckpt_metrics = load_checkpoint(CKPT_PATH, map_location=DEVICE)
    print(f"Loaded checkpoint: {CKPT_PATH}")
    if ckpt_metrics.get("epochs"):
        last = ckpt_metrics["epochs"][-1]
        print(f"  Trained for {len(ckpt_metrics['epochs'])} epochs  "
              f"| final test_acc={last.get('test_acc','?'):.4f}")
else:
    model, meta = train_quick(DEVICE, DATA_DIR, SEED)
    model, meta, ckpt_metrics = load_checkpoint(CKPT_PATH, map_location=DEVICE)

model.to(DEVICE).eval()
split = ensure_split(SUBSET_PATH)
print(f"Eval split: {len(split.indices)} samples (seed={split.seed})")

## 4 — Run IBP on all 100 samples

In [ ]:
_, test_ds = get_mnist_datasets(DATA_DIR)
sub_ds = Subset(test_ds, split.indices)
loader = DataLoader(sub_ds, batch_size=1, shuffle=False, num_workers=0)

ibp_rows = []
t0_ibp   = time.time()
model_cpu = model.to("cpu").eval()

for local_i, (x, y) in enumerate(loader):
    idx   = int(split.indices[local_i])
    label = int(y[0].item())
    status, margin = ibp_verify_point(model_cpu, x[0], label, EPS)
    ibp_rows.append({"index": idx, "y": label, "ibp_status": status, "ibp_margin": margin})

ibp_time = time.time() - t0_ibp
df_ibp   = pd.DataFrame(ibp_rows)
model.to(DEVICE)

n_ibp_verified = (df_ibp["ibp_status"] == "VERIFIED").sum()
print(f"IBP results on {len(df_ibp)} samples  (total time: {ibp_time:.2f}s  avg: {ibp_time/len(df_ibp)*1000:.1f}ms/sample)")
print(f"  VERIFIED:     {n_ibp_verified} / {len(df_ibp)} ({n_ibp_verified/len(df_ibp)*100:.1f}%)")
print(f"  INCONCLUSIVE: {len(df_ibp)-n_ibp_verified} / {len(df_ibp)} ({(len(df_ibp)-n_ibp_verified)/len(df_ibp)*100:.1f}%)")

## 5 — Run PGD on all 100 samples

In [ ]:
pgd_csv = f"results/pgd_{meta.run_name}_eps{EPS:.4f}.csv"

if Path(pgd_csv).exists():
    df_pgd = pd.read_csv(pgd_csv)
    pgd_time = df_pgd.get("pgd_time_s", pd.Series([0.])).sum()
    print(f"Loaded PGD results from {pgd_csv} ({len(df_pgd)} rows)")
else:
    print("Running PGD attack...")
    loader_pgd = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    pgd_rows   = []; t0_pgd = time.time()
    for batch_i, (x0, y) in enumerate(loader_pgd):
        x0, y = x0.to(DEVICE), y.to(DEVICE)
        with torch.no_grad():
            clean_pred = model(x0).argmax(1)
            clean_loss = F.cross_entropy(model(x0), y, reduction="none")
        x_adv = pgd_linf(model, x0, y, EPS, PGD_STEPS, STEP_SIZE, random_start=True)
        with torch.no_grad():
            adv_pred = model(x_adv).argmax(1)
            adv_loss = F.cross_entropy(model(x_adv), y, reduction="none")
        base = batch_i * BATCH_SIZE
        for i in range(y.shape[0]):
            pgd_rows.append({
                "index":      int(split.indices[base + i]),
                "y":          int(y[i]),
                "clean_pred": int(clean_pred[i]),
                "adv_pred":   int(adv_pred[i]),
                "success":    int(adv_pred[i]) != int(y[i]),
                "clean_loss": float(clean_loss[i]),
                "adv_loss":   float(adv_loss[i]),
            })
    pgd_time = time.time() - t0_pgd
    df_pgd = pd.DataFrame(pgd_rows)
    Path("results").mkdir(parents=True, exist_ok=True)
    df_pgd.to_csv(pgd_csv, index=False)
    print(f"PGD done in {pgd_time:.2f}s. Saved to {pgd_csv}")

asr        = df_pgd["success"].mean()
clean_acc  = (df_pgd["clean_pred"] == df_pgd["y"]).mean()
pgd_avg_ms = (pgd_time / len(df_pgd) * 1000) if pgd_time else 0
print(f"  Clean accuracy  : {clean_acc:.4f}")
print(f"  Attack success  : {asr:.4f} ({df_pgd['success'].sum()}/{len(df_pgd)})")

## 6 — Run MILP on first N samples

In [ ]:
NOTEBOOKS_RESULTS = Path("notebooks/results")
NOTEBOOKS_RESULTS.mkdir(parents=True, exist_ok=True)

milp_csv = NOTEBOOKS_RESULTS / f"milp_{meta.run_name}_eps{EPS:.4f}.csv"

if milp_csv.exists():
    df_milp_raw = pd.read_csv(milp_csv)
    df_milp     = df_milp_raw.head(MAX_MILP_SAMPLES).copy()
    milp_time   = df_milp["time_s"].sum()
    print(f"Loaded MILP results from {milp_csv} ({len(df_milp)} rows used)")
else:
    print(f"Running MILP verification on first {MAX_MILP_SAMPLES} samples...")
    milp_indices = split.indices[:MAX_MILP_SAMPLES]
    milp_sub     = Subset(test_ds, milp_indices)
    milp_loader  = DataLoader(milp_sub, batch_size=1, shuffle=False, num_workers=0)
    milp_rows    = []
    model_cpu    = model.to("cpu").eval()

    for li, (x, y) in enumerate(milp_loader):
        idx   = int(milp_indices[li])
        label = int(y[0])
        print(f"  [{li+1}/{MAX_MILP_SAMPLES}] idx={idx} label={label}", end=" ", flush=True)
        res = milp_verify_point(model_cpu, x[0], label, EPS, time_limit_s=TIME_LIMIT)
        print(f"-> {res.status.value}  margin={res.worst_margin}  t={res.solve_time:.2f}s")
        milp_rows.append({"index": idx, "y": label, "status": res.status.value,
                          "worst_margin": res.worst_margin, "time_s": res.solve_time,
                          "solver": res.solver_name})

    model.to(DEVICE)
    milp_time = sum(r["time_s"] for r in milp_rows)
    df_milp   = pd.DataFrame(milp_rows)
    df_milp.to_csv(milp_csv, index=False)
    print(f"MILP done in {milp_time:.2f}s. Saved to {milp_csv}")

n_verified = (df_milp["status"] == "VERIFIED").sum()
n_falsif   = (df_milp["status"] == "FALSIFIED").sum()
print(f"  VERIFIED : {n_verified}/{len(df_milp)}")
print(f"  FALSIFIED: {n_falsif}/{len(df_milp)}")
print(f"  Mean solve time: {df_milp['time_s'].mean():.2f}s/sample")

## 7 — Correctness Verification

We check all five theoretical guarantees and flag any violations.

In [ ]:
# Merge IBP and PGD on shared indices
df_joint = pd.merge(df_ibp, df_pgd[["index","y","clean_pred","adv_pred","success","clean_loss","adv_loss"]],
                    on=["index","y"], how="inner")

# Merge in MILP for the MILP subset
df_milp_m = df_milp.rename(columns={"status":"milp_status","worst_margin":"milp_margin","time_s":"milp_time_s"})
df_full   = pd.merge(df_joint, df_milp_m[["index","milp_status","milp_margin","milp_time_s"]],
                     on="index", how="left")

violations = []

# CHECK 1: PGD soundness
# success=True must mean adv_pred != y  (already in the data by construction,
# but we verify the CSV is internally consistent)
c1 = df_joint[df_joint["success"] & (df_joint["adv_pred"] == df_joint["y"])]
if len(c1):
    violations.append(f"CHECK 1 FAILED: {len(c1)} rows have success=True but adv_pred==y")
else:
    print("CHECK 1 PASS — PGD soundness: all 'success=True' samples have adv_pred != y")

# CHECK 2: IBP soundness — IBP VERIFIED => PGD fails
ibp_verified = df_joint[df_joint["ibp_status"]=="VERIFIED"]
c2 = ibp_verified[ibp_verified["success"]]
if len(c2):
    violations.append(f"CHECK 2 FAILED (IBP UNSOUND): {len(c2)} IBP-VERIFIED samples were broken by PGD")
else:
    print(f"CHECK 2 PASS — IBP soundness: 0/{len(ibp_verified)} IBP-VERIFIED samples broken by PGD")

# CHECK 3: PGD success => MILP FALSIFIED
df_m = df_full.dropna(subset=["milp_status"])
c3   = df_m[df_m["success"] & (df_m["milp_status"]!="FALSIFIED")]
if len(c3):
    violations.append(f"CHECK 3 FAILED: {len(c3)} PGD-success samples NOT marked MILP FALSIFIED")
else:
    pgd_in_milp = df_m["success"].sum()
    print(f"CHECK 3 PASS — PGD success => MILP FALSIFIED: {pgd_in_milp} PGD successes all MILP FALSIFIED")

# CHECK 4: IBP VERIFIED => MILP VERIFIED
ibp_v_m = df_m[df_m["ibp_status"]=="VERIFIED"]
c4 = ibp_v_m[ibp_v_m["milp_status"]!="VERIFIED"]
if len(c4):
    violations.append(f"CHECK 4 FAILED (IBP UNSOUND): {len(c4)} IBP-VERIFIED samples not MILP VERIFIED")
else:
    print(f"CHECK 4 PASS — IBP VERIFIED => MILP VERIFIED: {len(ibp_v_m)} samples consistent")

# CHECK 5: MILP VERIFIED => PGD fails
milp_v = df_m[df_m["milp_status"]=="VERIFIED"]
c5 = milp_v[milp_v["success"]]
if len(c5):
    violations.append(f"CHECK 5 FAILED (MILP UNSOUND): {len(c5)} MILP-VERIFIED samples broken by PGD")
else:
    print(f"CHECK 5 PASS — MILP VERIFIED => PGD fails: {len(milp_v)} samples consistent")

# CHECK 6: IBP conservatism — IBP non-verified rate >= MILP non-verified rate
ibp_nonv_rate  = (df_m["ibp_status"]!="VERIFIED").mean()
milp_nonv_rate = (df_m["milp_status"]!="VERIFIED").mean()
if ibp_nonv_rate < milp_nonv_rate - 0.001:
    violations.append(f"CHECK 6 FAILED: IBP non-verified rate ({ibp_nonv_rate:.3f}) < MILP ({milp_nonv_rate:.3f})")
else:
    print(f"CHECK 6 PASS — IBP conservatism: IBP non-verified={ibp_nonv_rate:.3f} >= MILP non-verified={milp_nonv_rate:.3f}")

print()
if violations:
    print("VIOLATIONS FOUND:")
    for v in violations: print(f"  !! {v}")
else:
    print("ALL CORRECTNESS CHECKS PASSED — methods are consistent with their theoretical guarantees.")

## 8 — Visualisations

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

matplotlib.rcParams.update({
    "figure.dpi": 110, "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
})
COLORS = {"PGD": "#e15759", "IBP": "#f28e2b", "MILP": "#4e79a7",
          "Verified": "#59a14f", "NonVerified": "#e15759", "Clean": "#76b7b2"}
Path("results/figures").mkdir(parents=True, exist_ok=True)
print("Plotting libraries ready")

### Figure 1 — Robustness rate by method (all 100 samples vs MILP subset)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Robustness Comparison (eps={EPS}, MLP 784-128-64-10)", fontsize=14, fontweight="bold")

# ── Left: all 100 samples (IBP + PGD) ────────────────────────────────────────
ax = axes[0]
ibp_ver_pct  = (df_ibp["ibp_status"]=="VERIFIED").mean() * 100
ibp_inc_pct  = 100 - ibp_ver_pct
pgd_rob_pct  = 100 - df_pgd["success"].mean() * 100
pgd_atk_pct  = df_pgd["success"].mean() * 100

methods = ["IBP\n(sound, incomplete)", "PGD\n(empirical attack)"]
robust  = [ibp_ver_pct,  pgd_rob_pct]
nonrob  = [ibp_inc_pct,  pgd_atk_pct]
x = range(len(methods))
bars1 = ax.bar(x, robust, color=COLORS["Verified"],  label="Robust / Verified",     width=0.5)
bars2 = ax.bar(x, nonrob, color=COLORS["NonVerified"], label="Non-robust / Inconclusive",
               bottom=robust, width=0.5, alpha=0.85)
for bar, val in zip(bars1, robust):
    ax.text(bar.get_x()+bar.get_width()/2, val/2, f"{val:.1f}%",
            ha="center", va="center", color="white", fontweight="bold", fontsize=11)
for bar, val, bot in zip(bars2, nonrob, robust):
    ax.text(bar.get_x()+bar.get_width()/2, bot+val/2, f"{val:.1f}%",
            ha="center", va="center", color="white", fontweight="bold", fontsize=11)
ax.set_xticks(list(x)); ax.set_xticklabels(methods)
ax.set_ylabel("% of samples (n=100)"); ax.set_ylim(0, 105)
ax.set_title("All 100 evaluation samples"); ax.legend(loc="upper right", fontsize=10)

# ── Right: MILP subset (IBP + PGD + MILP side-by-side) ───────────────────────
ax2 = axes[1]
sub_ibp  = df_full.dropna(subset=["milp_status"])
n_sub    = len(sub_ibp)

milp_v_pct  = (sub_ibp["milp_status"]=="VERIFIED").mean() * 100
milp_f_pct  = (sub_ibp["milp_status"]=="FALSIFIED").mean() * 100
ibp_v2_pct  = (sub_ibp["ibp_status"]=="VERIFIED").mean() * 100
ibp_i2_pct  = 100 - ibp_v2_pct
pgd_r2_pct  = 100 - sub_ibp["success"].mean() * 100
pgd_a2_pct  = sub_ibp["success"].mean() * 100

methods3 = ["IBP", "PGD", "MILP (exact)"]
rob3     = [ibp_v2_pct, pgd_r2_pct, milp_v_pct]
non3     = [ibp_i2_pct, pgd_a2_pct, milp_f_pct]
x3 = range(len(methods3))
b1 = ax2.bar(x3, rob3, color=COLORS["Verified"],     label="Robust / Verified",        width=0.5)
b2 = ax2.bar(x3, non3, color=COLORS["NonVerified"],  label="Non-robust / Falsified",
             bottom=rob3, width=0.5, alpha=0.85)
for bar, val in zip(b1, rob3):
    if val > 3:
        ax2.text(bar.get_x()+bar.get_width()/2, val/2, f"{val:.1f}%",
                 ha="center", va="center", color="white", fontweight="bold", fontsize=11)
for bar, val, bot in zip(b2, non3, rob3):
    if val > 3:
        ax2.text(bar.get_x()+bar.get_width()/2, bot+val/2, f"{val:.1f}%",
                 ha="center", va="center", color="white", fontweight="bold", fontsize=11)
ax2.set_xticks(list(x3)); ax2.set_xticklabels(methods3)
ax2.set_ylabel(f"% of samples (n={n_sub})"); ax2.set_ylim(0, 105)
ax2.set_title(f"MILP subset (first {n_sub} samples)"); ax2.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.savefig("results/figures/fig1_robustness_rates.png", bbox_inches="tight")
plt.show()
print("Figure 1 saved")

### Figure 2 — Per-sample method agreement heatmap (MILP subset)

In [ ]:
sub = df_full.dropna(subset=["milp_status"]).reset_index(drop=True)

# Encode: 1 = non-robust/falsified/success, 0 = robust/verified
mat = pd.DataFrame({
    "IBP\n(inconclusive=1)": (sub["ibp_status"] != "VERIFIED").astype(int).values,
    "PGD\n(attack success)": sub["success"].astype(int).values,
    "MILP\n(falsified=1)":   (sub["milp_status"] == "FALSIFIED").astype(int).values,
}, index=[f"s{int(r['index'])}" for _, r in sub.iterrows()])

fig, ax = plt.subplots(figsize=(6, max(4, len(sub)*0.45)))
cmap = matplotlib.colors.ListedColormap([COLORS["Verified"], COLORS["NonVerified"]])
sns.heatmap(mat, ax=ax, cmap=cmap, vmin=0, vmax=1, linewidths=0.5, linecolor="white",
            cbar=False, annot=mat.applymap(lambda v: "V" if v==0 else "X"),
            fmt="", annot_kws={"size": 10, "weight": "bold"})
ax.set_title(f"Per-sample verification results (green=robust, red=non-robust)\neps={EPS}",
             fontweight="bold")
ax.set_xlabel("Method"); ax.set_ylabel("Sample index")

green_patch = mpatches.Patch(color=COLORS["Verified"],    label="V = Verified / Robust")
red_patch   = mpatches.Patch(color=COLORS["NonVerified"], label="X = Falsified / Non-robust")
ax.legend(handles=[green_patch, red_patch], bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig("results/figures/fig2_per_sample_heatmap.png", bbox_inches="tight")
plt.show()
print("Figure 2 saved")

### Figure 3 — Computation time per method (log scale)

In [ ]:
n_all  = len(df_ibp)
n_milp = len(df_milp)

ibp_ms_per   = ibp_time  / n_all  * 1000
pgd_ms_per   = pgd_time  / n_all  * 1000 if pgd_time else (df_pgd.get("adv_loss", pd.Series([0])).count()/1*1000)
milp_ms_per  = df_milp["time_s"].mean() * 1000

fig, ax = plt.subplots(figsize=(8, 5))
methods_t = ["IBP", "PGD", "MILP (exact)"]
times_ms  = [ibp_ms_per, pgd_ms_per, milp_ms_per]
colors_t  = [COLORS["IBP"], COLORS["PGD"], COLORS["MILP"]]

bars = ax.bar(methods_t, times_ms, color=colors_t, width=0.5, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, times_ms):
    label = f"{val/1000:.1f}s" if val >= 1000 else f"{val:.1f}ms"
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.05,
            label, ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_yscale("log")
ax.set_ylabel("Time per sample (ms, log scale)")
ax.set_title("Average Computation Time per Sample per Method", fontweight="bold")
ax.set_ylim(bottom=0.1)

# Annotate the trade-off
ratio = milp_ms_per / max(ibp_ms_per, 1e-3)
ax.annotate(f"MILP is ~{ratio:.0f}x slower than IBP",
            xy=(2, milp_ms_per), xytext=(1.2, milp_ms_per*0.15),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=10, color="gray")

plt.tight_layout()
plt.savefig("results/figures/fig3_timing.png", bbox_inches="tight")
plt.show()
print("Figure 3 saved")

### Figure 4 — IBP worst-case margin vs MILP exact margin (MILP subset)

In [ ]:
sub2 = df_full.dropna(subset=["milp_status", "milp_margin"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Margin Analysis: IBP (over-approximation) vs MILP (exact)",
             fontweight="bold", fontsize=13)

# Left: scatter — IBP margin vs MILP exact margin
ax = axes[0]
color_map = sub2["milp_status"].map({"VERIFIED": COLORS["Verified"], "FALSIFIED": COLORS["NonVerified"]})
ax.scatter(sub2["milp_margin"], sub2["ibp_margin"],
           c=color_map, s=90, edgecolors="white", linewidth=0.8, zorder=3)
lim_min = min(sub2["milp_margin"].min(), sub2["ibp_margin"].min()) - 0.5
lim_max = max(sub2["milp_margin"].max(), sub2["ibp_margin"].max()) + 0.5
ax.plot([lim_min, lim_max], [lim_min, lim_max], "k--", alpha=0.3, label="y = x (equal margins)")
ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.axvline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("MILP exact worst-case margin (logit_c - logit_y)")
ax.set_ylabel("IBP worst-case margin (over-approximation)")
ax.set_title("IBP margin >= MILP margin (IBP is conservative)")
v_patch = mpatches.Patch(color=COLORS["Verified"],    label="MILP VERIFIED")
f_patch = mpatches.Patch(color=COLORS["NonVerified"], label="MILP FALSIFIED")
ax.legend(handles=[v_patch, f_patch, ax.lines[0]], fontsize=9)

# Right: margin gap bar chart
ax2 = axes[1]
sub2["margin_gap"] = sub2["ibp_margin"] - sub2["milp_margin"]
sample_labels = [f"s{int(r['index'])}" for _, r in sub2.iterrows()]
bar_colors2   = sub2["milp_status"].map({"VERIFIED": COLORS["Verified"], "FALSIFIED": COLORS["NonVerified"]})
ax2.bar(range(len(sub2)), sub2["margin_gap"].values, color=bar_colors2, width=0.7, edgecolor="white")
ax2.set_xticks(range(len(sub2))); ax2.set_xticklabels(sample_labels, rotation=45, ha="right", fontsize=9)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Sample"); ax2.set_ylabel("IBP margin - MILP margin (gap)")
ax2.set_title("IBP over-approximation gap per sample\n(larger = IBP is more conservative)")
ax2.legend(handles=[v_patch, f_patch], fontsize=9)

plt.tight_layout()
plt.savefig("results/figures/fig4_margin_analysis.png", bbox_inches="tight")
plt.show()
print(f"Mean IBP-MILP gap: {sub2['margin_gap'].mean():.4f}")
print(f"IBP margin >= MILP margin for all samples: {(sub2['ibp_margin'] >= sub2['milp_margin'] - 1e-4).all()}")

### Figure 5 — PGD loss distribution (clean vs adversarial)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("PGD Attack Analysis — Loss & Prediction Shift", fontweight="bold", fontsize=13)

# Left: overlapping histograms of clean vs adv loss
ax = axes[0]
bins = 25
ax.hist(df_pgd["clean_loss"], bins=bins, alpha=0.65, color=COLORS["Clean"],
        label=f"Clean loss (mean={df_pgd['clean_loss'].mean():.3f})", edgecolor="white")
ax.hist(df_pgd["adv_loss"],   bins=bins, alpha=0.65, color=COLORS["PGD"],
        label=f"Adversarial loss (mean={df_pgd['adv_loss'].mean():.3f})", edgecolor="white")
ax.set_xlabel("Cross-entropy loss"); ax.set_ylabel("Count")
ax.set_title(f"Clean vs adversarial cross-entropy loss\n(eps={EPS}, steps={PGD_STEPS})")
ax.legend(fontsize=10)

# Right: per-class attack success rate
ax2 = axes[1]
class_asr = df_pgd.groupby("y")["success"].agg(["mean","count"]).reset_index()
class_asr.columns = ["digit","asr","count"]
bar_c = ax2.bar(class_asr["digit"], class_asr["asr"]*100,
                color=COLORS["PGD"], width=0.7, edgecolor="white")
for bar, cnt in zip(bar_c, class_asr["count"]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"n={cnt}", ha="center", va="bottom", fontsize=9)
ax2.axhline(df_pgd["success"].mean()*100, color="black", linestyle="--",
            linewidth=1.2, label=f"Overall ASR = {df_pgd['success'].mean()*100:.1f}%")
ax2.set_xlabel("True digit class"); ax2.set_ylabel("Attack success rate (%)")
ax2.set_title("PGD attack success rate by digit class")
ax2.set_xticks(range(10)); ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig("results/figures/fig5_pgd_analysis.png", bbox_inches="tight")
plt.show()
print("Figure 5 saved")

### Figure 6 — Epsilon sensitivity: IBP verified rate and PGD attack success rate

In [ ]:
print("Running epsilon sweep (IBP + PGD)...")
eps_records = []
model_cpu   = model.to("cpu").eval()
loader_eps  = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for eps_val in EPS_SWEEP:
    # IBP — fast, run per-sample
    loader_single = DataLoader(sub_ds, batch_size=1, shuffle=False, num_workers=0)
    ibp_v = sum(
        1 for li, (x,y) in enumerate(loader_single)
        if ibp_verify_point(model_cpu, x[0], int(y[0]), eps_val)[0] == "VERIFIED"
    )
    ibp_rate = ibp_v / len(split.indices)

    # PGD — batched
    model_tmp = model.to(DEVICE)
    successes = 0
    for x0, y in DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0):
        x0, y = x0.to(DEVICE), y.to(DEVICE)
        x_adv = pgd_linf(model_tmp, x0, y, eps_val, PGD_STEPS, eps_val/4, random_start=True)
        with torch.no_grad():
            successes += int((model_tmp(x_adv).argmax(1) != y).sum())
    pgd_asr = successes / len(split.indices)

    eps_records.append({"eps": eps_val, "ibp_verified": ibp_rate, "pgd_robust": 1-pgd_asr})
    print(f"  eps={eps_val:.3f}  IBP_verified={ibp_rate:.3f}  PGD_robust={1-pgd_asr:.3f}")

model.to(DEVICE)
df_eps = pd.DataFrame(eps_records)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_eps["eps"], df_eps["ibp_verified"]*100,  "o-", color=COLORS["IBP"],
        linewidth=2, markersize=7, label="IBP Verified %")
ax.plot(df_eps["eps"], df_eps["pgd_robust"]*100,    "s-", color=COLORS["PGD"],
        linewidth=2, markersize=7, label="PGD Robust % (1 - ASR)")
ax.fill_between(df_eps["eps"],
                df_eps["pgd_robust"]*100, df_eps["ibp_verified"]*100,
                alpha=0.12, color="gray", label="Precision gap (IBP conservatism)")
ax.axvline(EPS, color="gray", linestyle=":", linewidth=1.5, label=f"eps={EPS} (main experiment)")
ax.set_xlabel("Perturbation radius (eps, L-inf)")
ax.set_ylabel("% of samples (n=100)")
ax.set_title("Epsilon Sensitivity: IBP Verified Rate vs PGD Empirical Robustness",
             fontweight="bold")
ax.legend(fontsize=10); ax.set_ylim(-2, 102)
ax.set_xticks(EPS_SWEEP); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/figures/fig6_epsilon_sensitivity.png", bbox_inches="tight")
plt.show()
print("Figure 6 saved")

### Figure 7 — Method disagreement breakdown (MILP subset)

In [ ]:
sub3 = df_full.dropna(subset=["milp_status"]).copy()
sub3["ibp_v"]   = (sub3["ibp_status"]  == "VERIFIED").astype(int)
sub3["pgd_r"]   = (~sub3["success"].astype(bool)).astype(int)
sub3["milp_v"]  = (sub3["milp_status"] == "VERIFIED").astype(int)

# Label each sample with its combined outcome
def classify(row):
    ib, pg, ml = int(row["ibp_v"]), int(row["pgd_r"]), int(row["milp_v"])
    if ib==1 and pg==1 and ml==1: return "All agree: VERIFIED"
    if ib==0 and pg==0 and ml==0: return "All agree: FALSIFIED"
    if ib==0 and pg==1 and ml==1: return "MILP+PGD robust, IBP inconclusive"
    if ib==0 and pg==0 and ml==1: return "MILP verified, PGD attacks, IBP inconclusive"
    if ib==0 and pg==1 and ml==0: return "PGD robust, IBP+MILP: not robust"
    return f"Other (ib={ib},pg={pg},ml={ml})"

sub3["agreement"] = sub3.apply(classify, axis=1)
counts = sub3["agreement"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
pal = [COLORS["Verified"], COLORS["NonVerified"], COLORS["IBP"], COLORS["MILP"], COLORS["PGD"], "#9c755f"]
wedges, texts, autotexts = ax.pie(
    counts.values, labels=None, autopct=lambda p: f"{p:.1f}%" if p > 4 else "",
    colors=pal[:len(counts)], startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5),
    pctdistance=0.75
)
for at in autotexts: at.set(fontsize=10, fontweight="bold", color="white")
ax.legend(wedges, [f"{lab} (n={cnt})" for lab, cnt in counts.items()],
          loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
ax.set_title(f"Method Agreement on {len(sub3)} samples (eps={EPS})", fontweight="bold", fontsize=13)

plt.tight_layout()
plt.savefig("results/figures/fig7_method_agreement.png", bbox_inches="tight")
plt.show()
print("Figure 7 saved")
print(counts.to_string())

## 9 — Insights & Summary

In [ ]:
sub_m = df_full.dropna(subset=["milp_status"])
n_sub = len(sub_m)

ibp_ver_all  = (df_ibp["ibp_status"] == "VERIFIED").mean()
pgd_asr_all  = df_pgd["success"].mean()
clean_acc    = (df_pgd["clean_pred"] == df_pgd["y"]).mean()
milp_ver_sub = (sub_m["milp_status"] == "VERIFIED").mean()
milp_fal_sub = (sub_m["milp_status"] == "FALSIFIED").mean()
ibp_gap      = (sub_m["ibp_status"] != "VERIFIED").mean() - milp_fal_sub

# Compute timing locally — safe even if Figure 3 cell was not run
_ibp_ms_per  = ibp_time / max(len(df_ibp), 1) * 1000
_milp_ms_per = sub_m["milp_time_s"].mean() * 1000 if n_sub > 0 else 0.0

print("=" * 65)
print("RESULTS SUMMARY")
print("=" * 65)
print(f"Model      : MnistMlp (784->128->64->10), run='{meta.run_name}'")
print(f"Dataset    : MNIST test set, fixed {len(split.indices)}-sample split (seed={split.seed})")
print(f"Epsilon    : {EPS} (L-inf)")
print()
print(f"Clean accuracy on subset    : {clean_acc*100:.1f}%")
print()
print("--- Method results (all 100 samples) ---")
print(f"PGD attack success rate     : {pgd_asr_all*100:.1f}%  (empirical non-robustness)")
print(f"PGD robust (1 - ASR)        : {(1-pgd_asr_all)*100:.1f}%")
print(f"IBP verified rate           : {ibp_ver_all*100:.1f}%  (sound over-approximation)")
print(f"IBP inconclusive rate       : {(1-ibp_ver_all)*100:.1f}%")
print()
print(f"--- Method results (MILP subset, n={n_sub}) ---")
print(f"MILP verified               : {milp_ver_sub*100:.1f}%  (exact, complete)")
print(f"MILP falsified              : {milp_fal_sub*100:.1f}%")
print(f"Mean MILP solve time        : {sub_m['milp_time_s'].mean():.2f}s / sample")
print()
print("--- Key insights ---")
print(f"1. CORRECTNESS: {6 - len(violations)} / 6 theoretical soundness checks passed."
      f" {'No violations.' if not violations else str(len(violations)) + ' violation(s)!'}")
print(f"2. IBP CONSERVATISM: IBP inconclusive rate exceeds MILP falsified rate by"
      f" {ibp_gap*100:.1f}pp on the MILP subset.")
print(f"   => These are samples IBP cannot certify but MILP proves are actually safe.")
print(f"3. PGD vs MILP: PGD ASR {'~=' if abs(pgd_asr_all - milp_fal_sub) < 0.1 else '!='}"
      f" MILP falsification rate on subset — PGD captures most real adversarial examples.")
print(f"4. SPEED TRADE-OFF: IBP ~{_milp_ms_per / max(_ibp_ms_per, 1e-3):.0f}x faster/sample"
      f" than MILP — use IBP for large-scale screening, MILP for exact certificates.")
print(f"5. THESIS RELEVANCE:")
print(f"   - PGD: fast empirical lower bound on vulnerability.")
print(f"   - IBP: fast certified upper bound on robustness (conservative).")
print(f"   - MILP: exact ground-truth certificate (slow, scalable only to small subsets).")
print()
print(f"Figures saved to: results/figures/")
print(f"MILP CSV saved to: notebooks/results/")

## 10 — Save All Results to `notebooks/results/`

Collects every result generated in this session and writes them to `notebooks/results/`.
MILP results are already written there from Section 6. This cell also saves the IBP and PGD
summary CSVs so all three method outputs live together in one place.

In [ ]:
OUT = Path("notebooks/results")
OUT.mkdir(parents=True, exist_ok=True)

# ── IBP results ───────────────────────────────────────────────────────────────
ibp_out = OUT / f"ibp_{meta.run_name}_eps{EPS:.4f}.csv"
df_ibp.to_csv(ibp_out, index=False)
print(f"IBP  results saved : {ibp_out}  ({len(df_ibp)} rows)")

# ── PGD results ───────────────────────────────────────────────────────────────
pgd_out = OUT / f"pgd_{meta.run_name}_eps{EPS:.4f}.csv"
df_pgd.to_csv(pgd_out, index=False)
print(f"PGD  results saved : {pgd_out}  ({len(df_pgd)} rows)")

# ── MILP results (already saved in Section 6, confirm path here) ──────────────
milp_out = OUT / f"milp_{meta.run_name}_eps{EPS:.4f}.csv"
if not milp_out.exists():
    df_milp.to_csv(milp_out, index=False)
print(f"MILP results saved : {milp_out}  ({len(df_milp)} rows)")

# ── Epsilon sweep (IBP + PGD sensitivity) ────────────────────────────────────
if "df_eps" in dir():
    eps_out = OUT / f"eps_sweep_{meta.run_name}.csv"
    df_eps.to_csv(eps_out, index=False)
    print(f"Eps sweep saved    : {eps_out}  ({len(df_eps)} rows)")

# ── Joint table (IBP + PGD + MILP merged) ────────────────────────────────────
joint_out = OUT / f"joint_{meta.run_name}_eps{EPS:.4f}.csv"
df_full.to_csv(joint_out, index=False)
print(f"Joint table saved  : {joint_out}  ({len(df_full)} rows, MILP subset has values)")

print()
print(f"All outputs in: {OUT.resolve()}")
print(f"Files: {[p.name for p in sorted(OUT.glob('*.csv'))]}")